# Week 6 — Agents

**Ships today:** `hr-assistant` v5 — it completes goals, not just questions.

Week 5 gave the model tools. A question needed one or two tool calls. Now try this:

> *"Apply for 1 day of casual leave next Monday and tell my manager."*

That is a **goal**. It needs a balance check, then a leave request, then a message to the manager, in the
right order, and the assistant must know when it is finished. Two of those steps **change real data**.

> **An agent is a loop. The model picks the next step, your code does it, and the loop stops when the goal is met.**

```
GOAL -> REASON -> ACT -> CHECK -> DONE
          ^         |       |
          +---------+-------+      (not met: go round again)
```

## What is an agent, and what is not

| Example | Agent? | Why |
|---|---|---|
| "What is the leave policy?" answered from 3 retrieved chunks | No | One lookup, one answer. A fixed pipeline |
| "Convert 100 USD to INR" with one tool call | No (week 5) | One tool, one answer |
| **"Apply for leave next Monday and tell my manager"** | **Yes** | Several steps, the model chooses the order, it must stop when done |
| A coding assistant that reads files, edits them, runs the tests and fixes what fails | **Yes** | Loops until the tests pass |
| A support bot that looks up an order, checks the refund policy, issues the refund, emails the customer | **Yes** | Reads and writes, decides as it goes |
| A research assistant that searches, reads results, searches again, then writes a summary | **Yes** | Each search depends on the last result |
| A nightly script that runs the same 5 steps in a fixed order | No | You chose the order, not the model. That is a workflow, and often the better choice |

**Rule of thumb:** if you can write the steps down in advance, write a script. Use an agent when the
steps depend on what the tools return.

| Step | What we add | New idea |
|---|---|---|
| 1 | three tools | two writes, and tools that return `Error:` text |
| 2 | the loop | reason, act, repeat. Read a full trace |
| 3 | the check | a second call: is the goal really met? |
| 4 | caps | give up cleanly on an impossible goal |
| 5 | the gate | writes wait for a human |
| 6 | idempotency | a retry cannot create a duplicate |

Nothing new is imported. It is the week-5 loop with four extra ideas.

## Setup

In [ ]:
try:                       # on networks that intercept TLS, use the OS trust store
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

from dotenv import load_dotenv, find_dotenv
import os, json
from datetime import date, timedelta
from openai import OpenAI

print(".env found at:", find_dotenv(usecwd=True) or "NOT FOUND")
load_dotenv(override=True)                    # .env wins over an old OPENAI_API_KEY already set in the shell or kernel
api_key = os.environ["OPENAI_API_KEY"].strip()
print("API key in use ends with:", "..." + api_key[-4:])   # compare with the end of the key in .env
client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"
TODAY = date.today()

NEXT_MON = TODAY + timedelta(days=1)          # the first Monday after today
while NEXT_MON.weekday() != 0:
    NEXT_MON += timedelta(days=1)
print("today:", TODAY.strftime("%A %Y-%m-%d"), "| next Monday:", NEXT_MON)

---
## Step 1 — Three tools: one read, two writes

Our "HR system" is two Python lists. In the project they are database tables.

A **read** changes nothing, so it is safe any time. A **write** changes the world and cannot be taken back.
We name the writes now, because the loop will treat them differently.

Every failure returns text starting `Error:` that says what to do next (week 5: a tool that raises ends
the request, a tool that returns an error gives the model something to react to).

In [2]:
EMPLOYEE = {"name": "Asha", "manager": "Ravi Menon", "casual": 4, "sick": 6, "earned": 12, "salary": 1_850_000}

LEAVE_REQUESTS = []      # our fake HR system
NOTIFICATIONS = []

def reset():
    LEAVE_REQUESTS.clear()
    NOTIFICATIONS.clear()

def get_leave_balance():
    used = sum(r["days"] for r in LEAVE_REQUESTS if r["type"] == "casual")
    return {"casual": EMPLOYEE["casual"] - used, "sick": EMPLOYEE["sick"], "earned": EMPLOYEE["earned"]}

def apply_leave(leave_type, start_date, days):
    start = date.fromisoformat(start_date)
    if start <= TODAY:
        return f"Error: start_date must be after today ({TODAY})"
    if start.weekday() >= 5:
        return f"Error: {start_date} is a {start.strftime('%A')}. Pick a weekday"
    left = get_leave_balance()[leave_type]
    if days > left:
        return f"Error: only {left} {leave_type} day(s) left, you asked for {days}"
    request = {"id": len(LEAVE_REQUESTS) + 1, "type": leave_type, "start": start_date, "days": days}
    LEAVE_REQUESTS.append(request)
    return {"request_id": request["id"], "status": "submitted"}

def notify_manager(request_id, note):
    if not any(r["id"] == request_id for r in LEAVE_REQUESTS):
        return f"Error: no leave request {request_id}. Call apply_leave first"
    NOTIFICATIONS.append({"to": EMPLOYEE["manager"], "request_id": request_id, "note": note})
    return {"sent_to": EMPLOYEE["manager"]}

print(apply_leave("casual", str(NEXT_MON), 9))      # more days than the balance
print(notify_manager(99, "hello"))                  # a request that does not exist

Error: only 4 casual day(s) left, you asked for 9
Error: no leave request 99. Call apply_leave first


Now describe them to the model. Look at the descriptions: they say **when** to use each tool and that the
writes **change real data**. A description is prompt text.

And look at what is missing: **no tool takes an employee id.** The server knows who is logged in. The model is never asked.

In [7]:
def tool(name, description, properties=None, required=None):
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": properties or {}, "required": required or []}}}

TOOLS = [
    tool("get_leave_balance", "Get the remaining casual, sick and earned leave days. Use it before applying for leave."),
    tool("apply_leave",
         "Submit a leave request. This WRITES to the HR system. If it returns an Error, read it and fix the input. "
         "Do not call it twice for the same dates.",
         {"leave_type": {"type": "string", "enum": ["casual", "sick", "earned"]},
          "start_date": {"type": "string", "description": "YYYY-MM-DD, a weekday after today"},
          "days": {"type": "integer"}},
         ["leave_type", "start_date", "days"]),
    tool("notify_manager",
         "Send the manager a short message about a leave request. This SENDS a message. "
         "Use it only after apply_leave succeeded, with the request_id it returned.",
         {"request_id": {"type": "integer"}, "note": {"type": "string"}},
         ["request_id", "note"]),
]

FUNCTIONS = {"get_leave_balance": get_leave_balance, "apply_leave": apply_leave, "notify_manager": notify_manager}
WRITES = {"apply_leave", "notify_manager"}      # the irreversible ones

def run_tool(name, args):
    try:
        result = FUNCTIONS[name](**args)
    except Exception as e:                      # a tool must never crash the loop
        return f"Error: {name} failed: {type(e).__name__}: {e}"
    return result if isinstance(result, str) else json.dumps(result)

---
## Step 2 — The loop

Four words. **Goal** is what the user asked. **Reason** is one model call: what next? **Act** is *your code*
running what the model asked for. **Check** asks whether the goal is met.

Below is the whole loop. It is week 5's `run_tools` plus a goal, a step cap and a printed trace.
`check_goal` and `approve` are placeholders that do nothing yet. Steps 3 and 5 replace them, and the loop itself never changes.

In [8]:
SYSTEM = f"""You are the HR assistant for Rebind Rise, talking to {EMPLOYEE['name']}. Today is {TODAY.strftime('%A, %Y-%m-%d')}.
You complete goals by calling tools. Before each tool call, write one short sentence saying why.
Check the leave balance before applying. If a tool returns an Error, read it, fix the input and try again;
if you cannot fix it, tell the user plainly. Never say something was done unless a tool result shows it."""

USAGE = []                                          # tokens per model call

def check_goal(goal, messages):                     # placeholder, replaced in step 3
    return True, ""

def approve(name, args):                            # placeholder, replaced in step 5
    return True

def run_agent(goal, max_steps=6):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": goal}]
    start = len(USAGE)
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS)      # REASON
        USAGE.append(response.usage.total_tokens)
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        print(f"[{step}] reason: {msg.content or '(just a tool call)'}")

        if msg.tool_calls:                                                                          # ACT
            for call in msg.tool_calls:
                name, args = call.function.name, json.loads(call.function.arguments)
                if name in WRITES and not approve(name, args):
                    result = "Error: the employee declined this action. Do not retry it. Tell them what was not done."
                else:
                    result = run_tool(name, args)
                messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
                print(f"    act   : {name}({args}) -> {result}")
            continue

        met, missing = check_goal(goal, messages)                                                   # CHECK
        print(f"[{step}] check : met={met} {missing}")
        if met:
            return {"status": "done", "answer": msg.content, "steps": step, "tokens": sum(USAGE[start:])}
        messages.append({"role": "user", "content": f"Not finished yet: {missing} Continue the work."})

    return {"status": "stopped", "answer": f"I stopped after {max_steps} steps without finishing.",
            "steps": max_steps, "tokens": sum(USAGE[start:])}

A question first. One read, then the answer:

In [5]:
reset()
r = run_agent("How many casual leave days do I have left?")
print("\nANSWER:", r["answer"], "|", r["steps"], "steps,", r["tokens"], "tokens")

AuthenticationError: Error code: 401 - {'error': {'message': 'Your API key has been invalidated.', 'type': None, 'code': 'token_invalidated', 'param': None}, 'status': 401}

Now a **goal**: several steps, two writes.

In [ ]:
reset()
GOAL = f"Apply for 1 day of casual leave on {NEXT_MON} and tell my manager."
r = run_agent(GOAL)
print("\nANSWER:", r["answer"], "|", r["steps"], "steps,", r["tokens"], "tokens")
print("LEAVE_REQUESTS:", LEAVE_REQUESTS)
print("NOTIFICATIONS :", NOTIFICATIONS)

Read the trace: reason, act, reason, act, reason, act, reason. **The model chose the order.** We never wrote
"balance first, then apply". We gave it a goal and tools.

It also reacts to errors. Ask for something that will fail and watch the next `reason` line:

In [ ]:
reset()
r = run_agent(f"Apply for 9 days of casual leave on {NEXT_MON}.")
print("\nANSWER:", r["answer"])

It read the error text and told the user, instead of crashing. That is **reflection**, and no special code
produced it: the failure came back as a tool result, so the next reason step knew about it.

Now the first problem. A user double-clicks, a request is retried, or the same goal runs twice:

In [ ]:
reset()
run_agent(GOAL)
print("\n----- same goal again -----\n")
run_agent(GOAL)
print("\nLEAVE_REQUESTS:", len(LEAVE_REQUESTS), "| messages to manager:", len(NOTIFICATIONS))

**Two leave requests for the same day, two emails to the manager.** The agent did nothing wrong. It followed the
goal twice. This is the danger of write tools: the agent works as designed and still does damage.
Steps 5 and 6 fix it. First, the rest of the loop.

---
## Step 3 — The check

Without it you have a chatbot in a `while` loop: it stops when the model *says* it is finished, and a model
will say that when it is not. So the check is a **second call with a different job**: compare the goal to the
**tool results**, not to what the final message claims.

First a transcript we made up, where the assistant claims success but nothing was written:

In [ ]:
CHECK_PROMPT = ("You are a strict reviewer. You get a GOAL, the TOOL RESULTS so far, and the agent's FINAL MESSAGE. "
                "Judge by the tool results, never by what the final message claims. "
                'Reply with JSON only: {"met": true or false, "missing": "what is still not done, or empty"}. '
                "met is true if the goal was completed, or if it could not be completed and the final message says so "
                "honestly. met is false if work remains that the agent could still do, or if the final message claims "
                "something the tool results do not show.")

def check_goal(goal, messages):
    results = [m["content"] for m in messages if m["role"] == "tool"]
    body = (f"GOAL: {goal}\n\nTOOL RESULTS:\n" + "\n".join(f"- {r}" for r in results)
            + f"\n\nFINAL MESSAGE:\n{messages[-1].get('content')}")
    response = client.chat.completions.create(
        model=MODEL, response_format={"type": "json_object"},
        messages=[{"role": "system", "content": CHECK_PROMPT}, {"role": "user", "content": body}])
    USAGE.append(response.usage.total_tokens)
    verdict = json.loads(response.choices[0].message.content)
    return verdict["met"], verdict["missing"]

made_up = [{"role": "user", "content": GOAL},
           {"role": "tool", "tool_call_id": "x", "content": '{"casual": 4, "sick": 6, "earned": 12}'},
           {"role": "assistant", "content": "All done! Your leave is booked and your manager has been told."}]
print(check_goal(GOAL, made_up))

The only tool result is a balance. No leave was applied for, nobody was told, so `met = False` and it says what
is missing. In the loop that becomes *"Not finished yet: ... Continue the work."* and the agent goes round again.
The check costs one extra call each time the agent thinks it is done. That is cheap next to a wrong "done".

In [ ]:
reset()
r = run_agent(GOAL)
print("\nANSWER:", r["answer"], "|", r["steps"], "steps,", r["tokens"], "tokens")

---
## Step 4 — Caps, and giving up cleanly

A loop can go wrong two ways: the goal is impossible, or the agent keeps retrying something that cannot work.
That is why `run_agent` is a `for` loop with `max_steps`, not `while True`. **The cap is not optional.**

An impossible goal first (there is no cancel tool):

In [ ]:
reset()
r = run_agent("Cancel my leave request from last week.")
print("\nSTATUS:", r["status"], "| ANSWER:", r["answer"])

Giving up cleanly means **telling the user what was not done**, not inventing success. Now the cap itself: a real goal, two steps allowed.

In [ ]:
reset()
r = run_agent(GOAL, max_steps=2)
print("\nSTATUS:", r["status"], "| ANSWER:", r["answer"])
print("LEAVE_REQUESTS:", LEAVE_REQUESTS)

In the project we cap **tokens** as well as steps, because one step that returns a huge tool result can cost more than five small ones.

---
## Step 5 — The approval gate

Right now a write runs the moment the model asks. Fine for a demo. Not for a payment, a company-wide email, or a
deleted record. So **ACT splits in two**:

| Kind | Examples | What the loop does |
|---|---|---|
| **Reversible** | read a balance, search a policy, draft a message | runs it |
| **Irreversible** | write a request, send a message, pay, delete | **asks a human first** |

`act` already calls `approve(name, args)` for anything in `WRITES`. We replace the placeholder. Here the human
is a function that says no, then yes:

In [ ]:
def approve(name, args):
    print(f"    GATE  : {name}({args})  ->  DECLINED")
    return False

reset()
r = run_agent(GOAL)
print("\nANSWER:", r["answer"])
print("LEAVE_REQUESTS:", LEAVE_REQUESTS, "| NOTIFICATIONS:", NOTIFICATIONS)

**Nothing was written.** The decline went back to the model as text, so it could explain to the user. Now the human says yes:

In [ ]:
def approve(name, args):
    print(f"    GATE  : {name}({args})  ->  APPROVED")
    return True

reset()
r = run_agent(GOAL)
print("\nLEAVE_REQUESTS:", LEAVE_REQUESTS, "| NOTIFICATIONS:", NOTIFICATIONS)

In a notebook the human is a function. **In the project it is a confirm box in the browser.** The idea is identical:
the gate is **code in `act`**, not a sentence in the prompt. "Please ask permission first" is a request to the
model. This is a guarantee.

---
## Step 6 — Idempotency: a retry changes nothing

The gate asks *"should this happen?"* It does not stop a write the user **did** approve from running twice
(a retry, a double click, the model calling the tool again). For that, every write gets an **idempotency key**:
a fingerprint of *what the write is*. If a row with that key exists, return it instead of making another.

We redefine the two write tools. The loop does not change.

In [ ]:
def apply_leave(leave_type, start_date, days):
    key = f"{leave_type}|{start_date}|{days}"
    existing = next((r for r in LEAVE_REQUESTS if r["key"] == key), None)
    if existing:                                    # a retry: same answer, no new row
        return {"request_id": existing["id"], "status": "submitted", "note": "already on file, nothing new created"}
    start = date.fromisoformat(start_date)
    if start <= TODAY:
        return f"Error: start_date must be after today ({TODAY})"
    if start.weekday() >= 5:
        return f"Error: {start_date} is a {start.strftime('%A')}. Pick a weekday"
    left = get_leave_balance()[leave_type]
    if days > left:
        return f"Error: only {left} {leave_type} day(s) left, you asked for {days}"
    request = {"id": len(LEAVE_REQUESTS) + 1, "type": leave_type, "start": start_date, "days": days, "key": key}
    LEAVE_REQUESTS.append(request)
    return {"request_id": request["id"], "status": "submitted"}

def notify_manager(request_id, note):
    if not any(r["id"] == request_id for r in LEAVE_REQUESTS):
        return f"Error: no leave request {request_id}. Call apply_leave first"
    if any(n["request_id"] == request_id for n in NOTIFICATIONS):     # one message per request
        return {"sent_to": EMPLOYEE["manager"], "note": "manager was already notified, nothing sent again"}
    NOTIFICATIONS.append({"to": EMPLOYEE["manager"], "request_id": request_id, "note": note})
    return {"sent_to": EMPLOYEE["manager"]}

FUNCTIONS["apply_leave"] = apply_leave
FUNCTIONS["notify_manager"] = notify_manager

The problem from step 2, again: the same goal, run twice.

In [ ]:
reset()
run_agent(GOAL)
print("\n----- same goal again -----\n")
run_agent(GOAL)
print("\nLEAVE_REQUESTS:", len(LEAVE_REQUESTS), "| messages to manager:", len(NOTIFICATIONS))

**One request, one message, however many times it runs.** The key comes from *what the write is* (type, date, days),
never from something the model invents. In the project the database enforces it with a `UNIQUE` column, so even a
bug in your own code cannot create a duplicate.

**Gate = should this happen. Idempotency = what if it happens again.** You need both.

---
## What you built

| Step | Idea |
|---|---|
| 1 | Reads run freely, writes change the world. Errors are text. No tool takes an employee id |
| 2 | The loop: reason, act, repeat. The model picks the order, your code acts. An error is just another tool result |
| 3 | The check compares the goal to the tool results, not to the model's claim |
| 4 | Caps are not optional. Give up with a plain sentence, never loop |
| 5 | The gate is code in `act`. A decline goes back to the model as text |
| 6 | An idempotency key is a fingerprint of the write. A retry returns the old row |

**Which model runs which step?** Reason and check are where quality decides everything, so a stronger or "thinking"
model may be worth its cost there. Reading a balance does not need one. Measure it on your own goals.

**Seeing inside a run.** The trace you printed is what teams call a *trace*, each line a *span*. Tools like
Langfuse, LangSmith and Phoenix collect them. You do not need one today. Know the word.

**Naming it.** The paper that named this pattern is called *ReAct* (reason, act, repeat). You wrote it before hearing the name.

**Measuring an agent.** Judge by the *state of the world* (did the right row get written, and only one?), plus steps
and tokens per goal. An agent that gets it right in 11 steps has a problem. You do this in the exercise.

### Now put it in the project
Open `excersize.md`. The same loop goes into `hr-assistant`, with the gate as a confirm box in the browser.